# Session 2: from a known rule to a learned rule
**A. Real-world problem.** A mass on a spring with friction is measured with a noisy
sensor. We want a rule `dx/dt = f(x)` that predicts where it goes next. Today:
(1) solve a *known* rule with Euler and RK4 and measure how accurate each is,
(2) fit an *unknown* rule from data with least squares,
(3) replace the rule by a small neural network with hand-written backpropagation,
(4) ask how much data and how little noise we need.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

A_TRUE = np.array([[0.0, 1.0],
                   [-4.0, -0.4]])      # x' = v, v' = -4 x - 0.4 v  (stiffness 4 per s^2, damping 0.4 per s)

## Step 1: Euler, RK4 and their order
One **Euler** step follows the arrow once: `x + h f(x)`. One **RK4** step samples the arrow
four times (start, two midpoints, end) and blends them with weights 1, 2, 2, 1 over 6.
The *error after a fixed time T* shrinks like `h^p`: p = 1 for Euler, p = 4 for RK4. On
log-log axes that is a straight line with slope p, so we measure p with a line fit.

In [ ]:
def true_field(x):
    return A_TRUE @ x


def euler_step(f, x, h):
    return x + h * f(x)


def rk4_step(f, x, h):
    k1 = f(x)
    k2 = f(x + 0.5 * h * k1)
    k3 = f(x + 0.5 * h * k2)
    k4 = f(x + h * k3)
    return x + h / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)


_LAM, _V = np.linalg.eig(A_TRUE)
_VI = np.linalg.inv(_V)


def exact_state(x0, t):
    """Exact solution x(t) = exp(A t) x0, from the eigenvalues of A."""
    return np.real(_V @ (np.exp(_LAM * t) * (_VI @ np.asarray(x0, float))))


def integrate(step, f, x0, h, t_end):
    x = np.asarray(x0, float)
    for _ in range(int(round(t_end / h))):
        x = step(f, x, h)
    return x


def error_at_T(step, h, t_end=2.0, x0=(1.0, 0.0)):
    return float(np.linalg.norm(integrate(step, true_field, x0, h, t_end) - exact_state(x0, t_end)))


def observed_order(step, hs, t_end=2.0):
    """Slope of log(error) against log(h)."""
    errs = [error_at_T(step, h, t_end) for h in hs]
    return float(np.polyfit(np.log(hs), np.log(errs), 1)[0])

## Step 2: data, derivatives and a least-squares fit
Sensor data: `K` runs from random starts, `n` samples each, spaced `dt` apart, with
Gaussian noise of size `sigma`. The derivative at a sample is estimated from its two
neighbours (central difference). If we *believe* `f(x) = A x` for an unknown 2x2 matrix
`A`, then finding A is linear least squares: minimise the sum of squares of
`x_dot - A x` over all samples.

In [ ]:
def make_data(K=10, n=20, dt=0.1, sigma=0.01, seed=3, box=1.5):
    """Array of shape (n, K, 2) with noisy positions and velocities."""
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(-box, box, (K, 2))
    X = np.array([np.real((_V @ (np.exp(_LAM * k * dt)[:, None] * (_VI @ x0.T))).T) for k in range(n)])
    return X + rng.normal(0.0, sigma, X.shape)


def central_difference(X, dt):
    """Return (states, derivative estimates) as rows; the first and last sample of each run are dropped."""
    return X[1:-1].reshape(-1, 2), ((X[2:] - X[:-2]) / (2 * dt)).reshape(-1, 2)


def fit_linear_field(X, dt):
    S, D = central_difference(X, dt)
    return np.linalg.lstsq(S, D, rcond=None)[0].T


def mean_fit_error(K, sigma, dt=0.1, n=20, trials=30):
    """Average Frobenius norm of (A_hat - A) over several random data sets."""
    return float(np.mean([np.linalg.norm(fit_linear_field(make_data(K, n, dt, sigma, seed=s), dt) - A_TRUE)
                          for s in range(trials)]))

## Step 3: a neural network as f
`f(x) = W2 tanh(W1 x + b1) + b2` with 16 hidden units: 16*2 + 16 + 2*16 + 2 = 82 numbers.
The **one-step loss** compares the network's output at each sample with the derivative
estimate from its neighbours. Backpropagation by hand, for the mean squared error
`L = mean((out - y)^2)`: with `r = out - y`, `g = 2 r / size`, `a = tanh(W1 x + b1)`,
`d = (g W2) * (1 - a^2)` we get `dL/dW2 = g^T a`, `dL/db2 = sum g`, `dL/dW1 = d^T x`,
`dL/db1 = sum d`. Adam then moves each number using a running average of its gradient.

In [ ]:
def init_mlp(width=16, seed=0):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0.0, 0.5, (width, 2)), "b1": np.zeros(width),
            "W2": rng.normal(0.0, 0.3, (2, width)), "b2": np.zeros(2)}


def mlp(P, x):
    return np.tanh(x @ P["W1"].T + P["b1"]) @ P["W2"].T + P["b2"]


def mlp_loss_grad(P, x, y):
    a = np.tanh(x @ P["W1"].T + P["b1"])
    r = a @ P["W2"].T + P["b2"] - y
    g = 2.0 * r / r.size
    d = (g @ P["W2"]) * (1.0 - a * a)
    return float(np.mean(r ** 2)), {"W2": g.T @ a, "b2": g.sum(axis=0), "W1": d.T @ x, "b1": d.sum(axis=0)}


def train_one_step(S, D, iters=3000, lr=0.01, width=16, seed=0):
    """Adam on the one-step loss. Returns (parameters, loss history)."""
    P = init_mlp(width, seed)
    m = {k: np.zeros_like(v) for k, v in P.items()}
    v2 = {k: np.zeros_like(v) for k, v in P.items()}
    hist = []
    for t in range(1, iters + 1):
        loss, G = mlp_loss_grad(P, S, D)
        hist.append(loss)
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * G[k]
            v2[k] = 0.999 * v2[k] + 0.001 * G[k] ** 2
            P[k] = P[k] - lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
    return P, hist


def field_rms_error(f_learned, box=1.5, n=500, seed=5):
    """RMS difference between a learned and the true field over random states in the box."""
    g = np.random.default_rng(seed).uniform(-box, box, (n, 2))
    return float(np.sqrt(np.mean((f_learned(g) - g @ A_TRUE.T) ** 2)))


def rollout_error(f_batch, x0=(1.2, -0.5), t_end=3.0, h=0.1):
    """Distance to the exact solution at t_end after integrating the learned field with RK4."""
    x = integrate(rk4_step, lambda z: f_batch(z[None, :])[0], x0, h, t_end)
    return float(np.linalg.norm(x - exact_state(x0, t_end)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    hs = np.array([0.2, 0.1, 0.05, 0.025, 0.0125])
    pe, pr = observed_order(euler_step, hs), observed_order(rk4_step, hs)
    print(f"observed order: Euler {pe:.2f}, RK4 {pr:.2f}")
    x1 = np.array([1.0, 0.0])
    print("one step of 0.1 s from (1, 0): Euler", np.round(euler_step(true_field, x1, 0.1), 4),
          " RK4", np.round(rk4_step(true_field, x1, 0.1), 4), " exact", np.round(exact_state(x1, 0.1), 4))

    fig, ax = plt.subplots(figsize=(6, 4.2))
    ax.loglog(hs, [error_at_T(euler_step, h) for h in hs], "o-", label=f"Euler, slope {pe:.2f}")
    ax.loglog(hs, [error_at_T(rk4_step, h) for h in hs], "s-", label=f"RK4, slope {pr:.2f}")
    ax.set_xlabel("step size h (s)")
    ax.set_ylabel("error at t = 2 s (distance in state space)")
    ax.set_title("Halving h: Euler error /2, RK4 error /16")
    ax.legend()
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(OUT / "order_loglog.png", dpi=120)

    X = make_data(10, 20, 0.1, 0.01)
    A_hat = fit_linear_field(X, 0.1)
    print("least-squares A_hat =\n", np.round(A_hat, 3), "\nerror", round(float(np.linalg.norm(A_hat - A_TRUE)), 3))
    Ks, sigmas = (1, 3, 10, 30), (0.01, 0.05, 0.2)
    table = np.array([[mean_fit_error(K, s) for K in Ks] for s in sigmas])
    print("fit error |A_hat - A| (rows sigma 0.01, 0.05, 0.2; columns K = 1, 3, 10, 30 runs):\n", np.round(table, 3))
    fig2, ax2 = plt.subplots(figsize=(6, 4.2))
    for row, s in zip(table, sigmas):
        ax2.loglog(Ks, row, "o-", label=f"noise sigma = {s}")
    ax2.axhline(0.1, color="red", ls="--", label="target error 0.1")
    ax2.set_xlabel("number of runs K (20 samples each)")
    ax2.set_ylabel("error |A_hat - A| (1/s)")
    ax2.set_title("More data helps until noise sets the floor")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "design_table.png", dpi=120)

    S, D = central_difference(X, 0.1)
    P, hist = train_one_step(S, D)
    e_nn = field_rms_error(lambda g: mlp(P, g))
    e_ls = field_rms_error(lambda g: g @ A_hat.T)
    print(f"network: {sum(v.size for v in P.values())} parameters, final loss {hist[-1]:.4f}")
    print(f"field RMS error: network {e_nn:.3f}, least squares {e_ls:.3f}")
    print(f"error after 3 s of prediction: network {rollout_error(lambda g: mlp(P, g)):.3f}, "
          f"least squares {rollout_error(lambda g: g @ A_hat.T):.3f}")

    g = np.linspace(-1.5, 1.5, 11)
    gx, gy = np.meshgrid(g, g)
    pts = np.stack([gx.ravel(), gy.ravel()], axis=1)
    fig3, ax3 = plt.subplots(1, 2, figsize=(10, 4.2))
    ax3[0].semilogy(hist)
    ax3[0].set_xlabel("Adam iteration")
    ax3[0].set_ylabel("one-step loss")
    ax3[0].set_title("The loss stops at the noise level")
    ax3[0].grid(alpha=0.3)
    ft, fl = pts @ A_TRUE.T, mlp(P, pts)
    ax3[1].quiver(pts[:, 0], pts[:, 1], ft[:, 0], ft[:, 1], color="0.65", angles="xy", label="true f")
    ax3[1].quiver(pts[:, 0], pts[:, 1], fl[:, 0], fl[:, 1], color="tab:blue", angles="xy", alpha=0.6, label="network f")
    ax3[1].plot(X[:, :, 0], X[:, :, 1], "k.", ms=2)
    ax3[1].set_xlabel("position x (m)")
    ax3[1].set_ylabel("velocity v (m/s)")
    ax3[1].set_title("Learned field over the data (dots)")
    fig3.tight_layout()
    fig3.savefig(OUT / "network_field.png", dpi=120)

## Where this stops
The loss above looks at one sample at a time. Errors in the field are never *compounded*
by the solver during training, and noisy neighbours bias the derivative estimates. Session 3
trains through the solver itself.

## Coding tasks
1. **Easy.** Run `error_at_T(euler_step, h)` and `error_at_T(rk4_step, h)` for h = 0.4 and print them. Which method is more accurate at h = 0.4? Predict the ratio of errors for h = 0.1 before you run it.
2. **Stretch.** Change the noise to `sigma = 0.05` in `make_data` and train the network again. Report the field error and the loss, and explain why the loss ends higher.
3. **Explain.** Explain why the one-step loss can be very small while predictions 3 seconds ahead are poor. (Hint: what does the loss not see?)

## Assessment
- **A. Conceptual.** With noise 0.2 the fit error stays above 0.3 even with 30 runs, while with noise 0.01 a handful of runs is enough. Why does more data not fix the noisy case?
- **B. Numerical.** The Euler error at 2 s is 0.575 for h = 0.1. Predict the error for h = 0.025 using the order p = 1, then compare with the measured value 0.128. What would an order-4 method predict from its error 4.3e-5 at h = 0.1?
- **C. Graph and simulation interpretation.** In the error graph, how do you read the order of a method, and what would a slope of 2 mean? What happens to RK4 if h becomes extremely small?
- **D. Design and optimisation challenge.** Design: the sensor noise can be 0.01, 0.05 or 0.2 and you can record K = 1, 3, 10 or 30 runs of 20 samples. You need a fit error (|A_hat - A|) below 0.1. Which combinations work, and which sensor and how many runs would you choose?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # --- hand calculation: the rule at (1, 0) and one step of each method
    f0 = L.true_field(np.array([1.0, 0.0]))
    checks.append(("hand calculation: at x = 1, v = 0 the rule gives dx/dt = 0 and dv/dt = -4", np.allclose(f0, [0.0, -4.0])))
    x1 = np.array([1.0, 0.0])
    eu, rk, ex = L.euler_step(L.true_field, x1, 0.1), L.rk4_step(L.true_field, x1, 0.1), L.exact_state(x1, 0.1)
    err_eu, err_rk = float(np.linalg.norm(eu - ex)), float(np.linalg.norm(rk - ex))
    checks.append((f"worked example: from (1, 0) one Euler step of 0.1 s gives (1.0, -0.4); RK4 gives (0.9803, -0.3895) = exact to 1e-5",
                   np.allclose(eu, [1.0, -0.4], atol=1e-12) and np.allclose(rk, [0.9803, -0.3895], atol=1e-4) and err_rk < 1e-5))
    checks.append((f"worked example: the Euler step is off by {err_eu:.3f} (about 0.022), the RK4 step by under 1e-5", abs(err_eu - 0.0223) < 5e-4))

    k1 = L.true_field(x1)
    k2 = L.true_field(x1 + 0.05 * k1)
    k3 = L.true_field(x1 + 0.05 * k2)
    k4 = L.true_field(x1 + 0.1 * k3)
    checks.append(("worked example: RK4 stages from (1, 0) are k1 = (0, -4), k2 = (-0.2, -3.92), k3 = (-0.196, -3.8816), k4 = (-0.38816, -3.766336)",
                   np.allclose(k1, [0, -4]) and np.allclose(k2, [-0.2, -3.92]) and np.allclose(k3, [-0.196, -3.8816]) and np.allclose(k4, [-0.38816, -3.766336])))

    # --- order of accuracy
    hs = np.array([0.2, 0.1, 0.05, 0.025, 0.0125])
    pe, pr = L.observed_order(L.euler_step, hs), L.observed_order(L.rk4_step, hs)
    checks.append((f"Euler is first order: measured slope {pe:.2f} is between 0.9 and 1.2", 0.9 < pe < 1.2))
    checks.append((f"RK4 is fourth order: measured slope {pr:.2f} is between 3.8 and 4.3", 3.8 < pr < 4.3))
    r_eu = L.error_at_T(L.euler_step, 0.1) / L.error_at_T(L.euler_step, 0.05)
    r_rk = L.error_at_T(L.rk4_step, 0.1) / L.error_at_T(L.rk4_step, 0.05)
    checks.append((f"halving h divides the Euler error by about 2 ({r_eu:.2f}) and the RK4 error by about 16 ({r_rk:.1f})", 1.8 < r_eu < 2.5 and 14 < r_rk < 20))
    e1, e4, r1 = L.error_at_T(L.euler_step, 0.1), L.error_at_T(L.euler_step, 0.025), L.error_at_T(L.rk4_step, 0.1)
    checks.append((f"Euler error at 2 s is {e1:.3f} for h = 0.1 and {e4:.3f} for h = 0.025 (a first-order guess says a quarter of the first: 0.144); RK4 at h = 0.1 gives {r1:.1e}",
                   abs(e1 - 0.575) < 0.002 and abs(e4 - 0.128) < 0.002 and 4.2e-5 < r1 < 4.4e-5))
    checks.append(("exact solution at t = 0 is the starting state", np.allclose(L.exact_state([1.0, 0.5], 0.0), [1.0, 0.5])))

    # --- derivatives and least squares
    t = 0.1 * np.arange(6)
    lin = np.stack([np.stack([3.0 * t, -2.0 * t], axis=1)], axis=1)       # (6, 1, 2): straight line
    S, D = L.central_difference(lin, 0.1)
    checks.append(("edge: central differences of a straight line give its slope exactly, and drop the two ends",
                   S.shape == (4, 2) and np.allclose(D, [3.0, -2.0])))
    X = L.make_data(10, 20, 0.1, 0.01)
    A_hat = L.fit_linear_field(X, 0.1)
    e_fit = float(np.linalg.norm(A_hat - L.A_TRUE))
    checks.append((f"least squares recovers A to within 0.06 with 10 runs and noise 0.01 (error {e_fit:.3f})", e_fit < 0.06))
    checks.append(("the fitted A has the structure of the truth: upper right near 1 and lower left near -4",
                   abs(A_hat[0, 1] - 1.0) < 0.05 and abs(A_hat[1, 0] + 4.0) < 0.2))

    # --- design problem: how much data and how little noise
    e_k10 = L.mean_fit_error(10, 0.05)
    e_k1 = L.mean_fit_error(1, 0.05)
    e_s1 = L.mean_fit_error(1, 0.01)
    e_k3 = L.mean_fit_error(3, 0.05)
    checks.append((f"design: noise 0.01 with a single run already passes (error {e_s1:.3f}); noise 0.05 with 3 runs just passes on average (error {e_k3:.3f}, between 0.08 and 0.1)", e_s1 < 0.1 and 0.08 < e_k3 < 0.1))
    e_n30 = L.mean_fit_error(30, 0.2)
    e_n10 = L.mean_fit_error(10, 0.2)
    checks.append((f"design: noise 0.05 with 10 runs reaches error {e_k10:.3f}, below the target 0.1", e_k10 < 0.1))
    checks.append((f"design: noise 0.05 with only 1 run fails the target (error {e_k1:.3f})", e_k1 > 0.1))
    checks.append((f"design: at noise 0.2 even 30 runs leave error {e_n30:.2f} (above 0.3) and gain under 10 percent over 10 runs ({e_n10:.2f})",
                   e_n30 > 0.3 and e_n30 > 0.9 * e_n10))

    # --- the network: gradient check, then training
    P = L.init_mlp(5, seed=2)
    S2 = np.random.default_rng(0).normal(size=(7, 2))
    D2 = np.random.default_rng(1).normal(size=(7, 2))
    _, G = L.mlp_loss_grad(P, S2, D2)
    worst = 0.0
    for key in P:
        idx = (0,) * P[key].ndim
        e = 1e-6
        P[key][idx] += e
        lp = L.mlp_loss_grad(P, S2, D2)[0]
        P[key][idx] -= 2 * e
        lm = L.mlp_loss_grad(P, S2, D2)[0]
        P[key][idx] += e
        fd = (lp - lm) / (2 * e)
        worst = max(worst, abs(G[key][idx] - fd) / (abs(fd) + 1e-12))
    checks.append((f"hand-written backpropagation matches finite differences (worst relative gap {worst:.1e})", worst < 1e-6))
    checks.append(("the network has 82 parameters at width 16", sum(v.size for v in L.init_mlp(16).values()) == 82))
    S, D = L.central_difference(X, 0.1)
    Pn, hist = L.train_one_step(S, D)
    checks.append((f"training: the one-step loss falls from {hist[0]:.2f} to {hist[-1]:.4f} (below 0.01)", hist[-1] < 0.01 < hist[0]))
    e_nn = L.field_rms_error(lambda g: L.mlp(Pn, g))
    e_ls = L.field_rms_error(lambda g: g @ A_hat.T)
    checks.append((f"the network's field is within 0.15 RMS of the truth (got {e_nn:.3f})", e_nn < 0.15))
    checks.append((f"for a linear truth the 4-number least-squares fit is more accurate ({e_ls:.3f}) than the 82-number network", e_ls < e_nn))
    r_nn = L.rollout_error(lambda g: L.mlp(Pn, g))
    checks.append((f"predicting 3 s ahead with the network and RK4 is within 0.2 of the truth (got {r_nn:.3f})", r_nn < 0.2))
    checks.append(("edge: an untrained network with zero weights gives zero field",
                   float(np.abs(L.mlp({"W1": np.zeros((4, 2)), "b1": np.zeros(4), "W2": np.zeros((2, 4)), "b2": np.zeros(2)}, np.ones((3, 2)))).max()) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
